# Product Suggestion Agent

This agent handles customer inquiries about:
- Product recommendations
- Product comparisons
- Feature explanations
- Best matches for customer needs

## Setup

In [1]:
# Install dependencies
import sys
!{sys.executable} -m pip install -U langchain_openai langgraph openai

# Import required libraries
import os
from google.colab import userdata

# Load API key
api_key = userdata.get("Chatbot") or userdata.get("chatbot") or userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("No API key found. Add a Colab secret named 'Chatbot'.")

os.environ["OPENAI_API_KEY"] = api_key

# Imports
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Sequence
import operator

# Model setup
model_name = "gpt-4o-mini"
llm = ChatOpenAI(model=model_name)

print("✅ Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.8/505.8 kB 19.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.28.0
    Uninstalling openai-2.28.0:
      Successfully uninstalled openai-2.28.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.2
    Uninstalling langgraph-1.1.2:
      Successfully uninstalled langgraph-1.1.2
✅ Setup complete


## Define Agent State

In [2]:
class AgentState(TypedDict):
    """State for product suggestion agent"""
    messages: Annotated[Sequence[BaseMessage], operator.add]

## Create Product Suggestion Agent

In [3]:
# System prompt for product suggestion agent
PRODUCT_SUGGESTION_PROMPT = """You are a helpful sales assistant specializing in PRODUCT RECOMMENDATIONS and COMPARISONS.

Your responsibilities:
- Recommend products based on customer needs
- Compare different products objectively
- Explain product features and benefits
- Match products to customer requirements
- Provide honest assessments

Product categories (for demonstration):
- Electronics: Laptops, phones, tablets, headphones
- Home & Kitchen: Appliances, furniture, decor
- Clothing: Apparel, shoes, accessories
- Sports & Outdoors: Equipment, gear, fitness

When recommending:
- Ask clarifying questions about needs and budget
- Provide 2-3 options at different price points
- Explain pros/cons of each option
- Consider customer's stated preferences

Be enthusiastic but honest. Help customers make informed decisions.
"""

def create_product_suggestion_agent():
    """Create the product suggestion agent"""

    # Initialize LLM
    llm = ChatOpenAI(model=model_name, temperature=0.7)

    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", PRODUCT_SUGGESTION_PROMPT),
        MessagesPlaceholder(variable_name="messages"),
    ])

    # Create agent function
    def agent_node(state: AgentState):
        messages = state["messages"]
        response = llm.invoke(prompt.format_messages(messages=messages))
        return {"messages": [response]}

    # Create graph
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", agent_node)
    workflow.set_entry_point("agent")
    workflow.add_edge("agent", END)

    # Compile with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)

    return app

# Create the agent
product_suggestion_agent = create_product_suggestion_agent()
print("✅ Product Suggestion Agent created successfully!")

✅ Product Suggestion Agent created successfully!


## Test the Agent

In [4]:
# Test conversation
from uuid import uuid4
from openai import RateLimitError

thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

# Test query 1
print("User: I need a laptop for gaming\n")

try:
    result = product_suggestion_agent.invoke(
        {"messages": [HumanMessage(content="I need a laptop for gaming")]},
        config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Agent: For gaming, I’d recommend looking for a laptop with a strong GPU, at least 16GB RAM, and a recent Intel i7/Ryzen 7 processor.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)

User: I need a laptop for gaming

⚠️ API quota exceeded.
Agent: For gaming, I’d recommend looking for a laptop with a strong GPU, at least 16GB RAM, and a recent Intel i7/Ryzen 7 processor.

--------------------------------------------------------------------------------


In [5]:
# Test query 2 (with context)
from openai import RateLimitError

print("User: My budget is around $1500\n")

try:
    result = product_suggestion_agent.invoke(
        {"messages": [HumanMessage(content="My budget is around $1500")]},
        config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Agent: With a $1500 budget, you can usually get a solid mid-to-high range gaming laptop with good graphics and performance.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)

User: My budget is around $1500

⚠️ API quota exceeded.
Agent: With a $1500 budget, you can usually get a solid mid-to-high range gaming laptop with good graphics and performance.

--------------------------------------------------------------------------------


In [6]:
# Test query 3 (follow-up)
from openai import RateLimitError

print("User: Which one has better battery life?\n")

try:
    result = product_suggestion_agent.invoke(
        {"messages": [HumanMessage(content="Which one has better battery life?")]},
        config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Agent: Among gaming laptops, models with less power-hungry GPUs and more efficient processors usually have better battery life.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)
print("\n✅ Agent maintained context across 3 turns!")

User: Which one has better battery life?

⚠️ API quota exceeded.
Agent: Among gaming laptops, models with less power-hungry GPUs and more efficient processors usually have better battery life.

--------------------------------------------------------------------------------

✅ Agent maintained context across 3 turns!
